<a href="https://colab.research.google.com/github/Selma-2024/spam-email-detector/blob/main/EMAILspam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import string
import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud
nltk.download('stopwords')

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import files

# Upload the dataset file
uploaded = files.upload()

# Assuming the file uploaded is 'spam_ham_dataset.csv'
# If the file has a different name, please update the filename accordingly.

After uploading the file, you can now run the cell `sR3StdmDd974` again to load the data.

In [ ]:
data = pd.read_csv('spam_ham_dataset.csv')
data.head()
data.shape
sns.countplot(x='label', data=data)
plt.show()

In [ ]:
ham_msg = data[data['label'] == 'ham']
spam_msg = data[data['label'] == 'spam']

# Downsample Ham emails to match the number of Spam emails
ham_msg_balanced = ham_msg.sample(n=len(spam_msg), random_state=42)

# Combine balanced data
balanced_data = pd.concat([ham_msg_balanced, spam_msg]).reset_index(drop=True)

# Visualize the balanced dataset
sns.countplot(x='label', data=balanced_data)
plt.title("Balanced Distribution of Spam and Ham Emails")
plt.xticks(ticks=[0, 1], labels=['Ham (Not Spam)', 'Spam'])
plt.show()

In [ ]:
balanced_data['text'] = balanced_data['text'].str.replace('Subject', '')
balanced_data.head()

In [ ]:
punctuations_list = string.punctuation
def remove_punctuations(text):
    temp = str.maketrans('', '', punctuations_list)
    return text.translate(temp)

balanced_data['text']= balanced_data['text'].apply(lambda x: remove_punctuations(x))
balanced_data.head()

In [ ]:
def remove_stopwords(text):
    stop_words = stopwords.words('english')

    imp_words = []

    # Storing the important words
    for word in str(text).split():
        word = word.lower()

        if word not in stop_words:
            imp_words.append(word)

    output = " ".join(imp_words)

    return output


balanced_data['text'] = balanced_data['text'].apply(lambda text: remove_stopwords(text))
balanced_data.head()

In [ ]:
def plot_word_cloud(data, typ):
    email_corpus = " ".join(data['text'])
    wc = WordCloud(background_color='black', max_words=100, width=800, height=400).generate(email_corpus)
    plt.figure(figsize=(7, 7))
    plt.imshow(wc, interpolation='bilinear')
    plt.title(f'WordCloud for {typ} Emails', fontsize=15)
    plt.axis('off')
    plt.show()

plot_word_cloud(balanced_data[balanced_data['label'] == 'ham'], typ='Non-Spam')
plot_word_cloud(balanced_data[balanced_data['label'] == 'spam'], typ='Spam')

In [ ]:
train_X, test_X, train_Y, test_Y = train_test_split(
    balanced_data['text'], balanced_data['label'], test_size=0.2, random_state=42
)

tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_X)

train_sequences = tokenizer.texts_to_sequences(train_X)
test_sequences = tokenizer.texts_to_sequences(test_X)

max_len = 100  # Maximum sequence length
train_sequences = pad_sequences(train_sequences, maxlen=max_len, padding='post', truncating='post')
test_sequences = pad_sequences(test_sequences, maxlen=max_len, padding='post', truncating='post')

train_Y = (train_Y == 'spam').astype(int)
test_Y = (test_Y == 'spam').astype(int)

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=32, input_length=max_len),
    tf.keras.layers.LSTM(16),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')  # Output layer
])

model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

In [ ]:
es = EarlyStopping(patience=3, monitor='val_accuracy', restore_best_weights=True)
lr = ReduceLROnPlateau(patience=2, monitor='val_loss', factor=0.5, verbose=0)

history = model.fit(
    train_sequences, train_Y,
    validation_data=(test_sequences, test_Y),
    epochs=20,
    batch_size=32,
    callbacks=[lr, es]
)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_sequences, test_Y)
print('Test Loss :',test_loss)
print('Test Accuracy :',test_accuracy)

In [ ]:
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.show()

In [ ]:
import pickle
from tensorflow.keras.models import save_model

# Save Keras model
model.save('spam_model.h5')

# Save tokenizer (NOT vectorizer)
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("✅ Model and tokenizer saved!")

In [ ]:
from google.colab import files

files.download('spam_model.h5')
files.download('tokenizer.pkl')

In [ ]:
# Check what output your model gives
import numpy as np

sample = ["Congratulations! You won a free prize click now"]
seq = tokenizer.texts_to_sequences(sample)
padded = pad_sequences(seq, maxlen=100)  # what maxlen did you use?
pred = model.predict(padded)
print("Raw prediction:", pred)
print("Shape:", pred.shape)

In [ ]:
pip install tensorflow scikit-learn numpy

In [ ]:
import imaplib

EMAIL    = "abc@gmail.com"   # ← your exact gmail
PASSWORD = "abcdefghijklmnop"      # ← paste WITHOUT spaces

try:
    mail = imaplib.IMAP4_SSL('imap.gmail.com', 993)
    mail.login(EMAIL, PASSWORD)
    print("✅ Login successful!")
    mail.logout()
except Exception as e:
    print(f"❌ Failed: {e}")

In [ ]:
# ════════════════════════════════════════════
#     SPAM DETECTOR — KERAS + IMAP IN COLAB
# ════════════════════════════════════════════

import imaplib
import email
import pickle
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ────────────────────────────────────────────
# SECTION 1: YOUR DETAILS
# ────────────────────────────────────────────
EMAIL_ADDRESS = "selmaannasaju22@gmail.com"   # ← your Gmail
APP_PASSWORD  = "nlbj gsvb kxhh fupq"      # ← app password without spaces
EMAIL_COUNT   = 10                # ← number of emails to check
MAX_LENGTH    = 100                     # ← ⚠️ replace with your actual max_length


# ────────────────────────────────────────────
# SECTION 2: LOAD MODEL AND TOKENIZER
# ────────────────────────────────────────────

model     = load_model('spam_model.h5')
with open('tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

print("✅ Model and tokenizer loaded!")


# ────────────────────────────────────────────
# SECTION 3: CONNECT TO GMAIL
# ────────────────────────────────────────────

mail = imaplib.IMAP4_SSL('imap.gmail.com', 993)
mail.login(EMAIL_ADDRESS, APP_PASSWORD)
print("✅ Connected to Gmail!")


# ────────────────────────────────────────────
# SECTION 4: FETCH EMAILS
# ────────────────────────────────────────────

mail.select('inbox')
status, messages = mail.search(None, 'ALL')
email_ids  = messages[0].split()
latest_ids = email_ids[-EMAIL_COUNT:]  # get latest N emails

emails = []
for eid in reversed(latest_ids):
    status, msg_data = mail.fetch(eid, '(RFC822)')
    raw_email        = msg_data[0][1]
    msg              = email.message_from_bytes(raw_email)

    subject = msg['subject'] or 'No Subject'
    sender  = msg['from']    or 'Unknown'

    # Extract body
    body = ''
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == 'text/plain':
                body = part.get_payload(decode=True).decode('utf-8', errors='ignore')
                break
    else:
        body = msg.get_payload(decode=True).decode('utf-8', errors='ignore')

    emails.append({
        'id'     : eid,
        'subject': subject,
        'sender' : sender,
        'body'   : body
    })

print(f"✅ Fetched {len(emails)} emails!\n")


# ────────────────────────────────────────────
# SECTION 5: PREDICT AND DELETE SPAM
# ────────────────────────────────────────────

spam_count = 0
safe_count = 0

print("=" * 55)
print("         SCANNING EMAILS FOR SPAM")
print("=" * 55 + "\n")

for e in emails:

    # Combine subject and body
    full_text = e['subject'] + ' ' + e['body']

    # Tokenize and pad
    seq    = tokenizer.texts_to_sequences([full_text])
    padded = pad_sequences(seq, maxlen=MAX_LENGTH)

    # Predict
    prediction = model.predict(padded, verbose=0)[0][0]

    is_spam = prediction > 0.5  # above 0.5 = spam

    if is_spam:
        print(f"🚨 SPAM     | Score: {prediction:.2f}")
        print(f"            | From: {e['sender'][:45]}")
        print(f"            | Subject: {e['subject'][:45]}")

        # Move to trash
        mail.store(e['id'], '+FLAGS', '\\Deleted')
        mail.expunge()
        print(f"            | 🗑️  Deleted!\n")
        spam_count += 1

    else:
        print(f"✅ NOT SPAM | Score: {prediction:.2f}")
        print(f"            | From: {e['sender'][:45]}")
        print(f"            | Subject: {e['subject'][:45]}\n")
        safe_count += 1


# ────────────────────────────────────────────
# SECTION 6: SUMMARY
# ────────────────────────────────────────────

print("=" * 55)
print("📊 SUMMARY")
print("=" * 55)
print(f"   Total emails checked : {len(emails)}")
print(f"   Spam deleted         : {spam_count}")
print(f"   Safe emails          : {safe_count}")
print("=" * 55)

mail.logout()
print("\n🔒 Logged out from Gmail.")